# 03 — Local LLM (Zero-Shot & Few-Shot) 5-Fold CV

Evaluate `SmolLM2-1.7B-Instruct` locally on the **same CV folds** as BERT.

**Features**
- Responses cached under `results/llm/cache/` (safe to stop and resume)
- Zero-shot and few-shot prompting with JSON output parsing

**Tip:** Run one fold at a time (`FOLDS_TO_RUN = [0]`) — full run is slow (~hours per fold).

In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import LLM_MODEL_NAME, N_FOLDS, RESULTS_DIR
from src.data_utils import get_fold_dataframes, prepare_dataset
from src.llm_inference import LocalLLMClassifier, evaluate_llm_fold
from src.metrics import summarize_across_folds
from src.results_io import save_all_fold_metrics, save_fold_metrics, save_summary_json

# Process one fold at a time; cached responses are reused on restart
FOLDS_TO_RUN = [0]
MODES_TO_RUN = ["zero_shot", "few_shot"]

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    raise RuntimeError(
        "CUDA not available. Select kernel 'Python 3.12 (fuzzy-final)'."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device} ({torch.cuda.get_device_name(0)})")
print(f"LLM: {LLM_MODEL_NAME}")
print(f"Folds: {FOLDS_TO_RUN}, modes: {MODES_TO_RUN}")

Project root: C:\Users\yusuf\OneDrive\Desktop\vscode\FuzzyLogic\final
Device: cuda (NVIDIA GeForce RTX 5060)
LLM: HuggingFaceTB/SmolLM2-1.7B-Instruct
Folds: [0], modes: ['zero_shot', 'few_shot']


## 1. Load data and initialize local LLM

In [2]:
df, folds = prepare_dataset()
fold_dfs = get_fold_dataframes(df, folds)

classifier = LocalLLMClassifier(device=device)
print("LLM loaded.")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

c:\Users\yusuf\OneDrive\Desktop\vscode\FuzzyLogic\final\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yusuf\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-1.7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LLM loaded.


## 2. Run zero-shot and few-shot on each fold

Cache files: `results/llm/cache/fold_{N}_{mode}.json`

In [3]:
llm_results = []

for fold_idx in FOLDS_TO_RUN:
    test_df = fold_dfs[fold_idx]["test"]
    for mode in MODES_TO_RUN:
        print(f"\n=== Fold {fold_idx} | {mode} ===")
        result = evaluate_llm_fold(
            classifier=classifier,
            fold_idx=fold_idx,
            test_df=test_df,
            mode=mode,
        )
        save_fold_metrics(
            fold_idx,
            f"llm_{mode}",
            result.metrics,
            output_dir=RESULTS_DIR / "llm" / mode,
        )
        llm_results.append(
            {
                "fold": fold_idx,
                "mode": mode,
                "metrics": result.metrics,
            }
        )
        print(f"macro F1: {result.metrics['macro_f1']:.4f}")
        print(f"micro F1: {result.metrics['micro_f1']:.4f}")


=== Fold 0 | zero_shot ===


LLM zero_shot:   0%|          | 0/5000 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers

macro F1: 0.1385
micro F1: 0.1538

=== Fold 0 | few_shot ===


LLM few_shot:   0%|          | 0/5000 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

macro F1: 0.1747
micro F1: 0.1693


## 3. Summarize results

In [4]:
for mode in MODES_TO_RUN:
    mode_results = [r for r in llm_results if r["mode"] == mode]
    if not mode_results:
        continue
    save_all_fold_metrics(
        [{"fold": r["fold"], "metrics": r["metrics"]} for r in mode_results],
        method=f"llm_{mode}",
        output_dir=RESULTS_DIR / "llm" / mode,
    )
    print(f"\n--- {mode} ---")
    display(summarize_across_folds([r["metrics"] for r in mode_results]))


--- zero_shot ---


,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
label,,,,,,
identity_hate,0.0270,NaN,0.9318,NaN,0.052400,NaN
insult,0.1474,NaN,0.9150,NaN,0.253900,NaN
obscene,0.1573,NaN,0.9094,NaN,0.268200,NaN
severe_toxic,0.0311,NaN,0.9600,NaN,0.060300,NaN
threat,0.0086,NaN,0.8667,NaN,0.016900,NaN
toxic,0.0984,NaN,0.9937,NaN,0.179000,NaN
macro_f1,NaN,NaN,NaN,NaN,0.138480,0.0
micro_f1,NaN,NaN,NaN,NaN,0.153846,0.0



--- few_shot ---


,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
label,,,,,,
identity_hate,0.0585,NaN,0.8182,NaN,0.109300,NaN
insult,0.1187,NaN,0.8866,NaN,0.209400,NaN
obscene,0.3419,NaN,0.7019,NaN,0.459800,NaN
severe_toxic,0.0252,NaN,0.9600,NaN,0.049100,NaN
threat,0.0136,NaN,1.0000,NaN,0.026700,NaN
toxic,0.1080,NaN,0.9478,NaN,0.193900,NaN
macro_f1,NaN,NaN,NaN,NaN,0.174709,0.0
micro_f1,NaN,NaN,NaN,NaN,0.169288,0.0
